# CMP611 Progress-2 Colab Notebook (Promoter-Focused + External Promoter Dataset)

Bu notebook, **human_ocr_ensembl kullanmadan** çalışır.

Kapsam:
1. `project.zip` yükle
2. Ortam kur
3. GUE dataset indir
4. GUE task splitlerini üret (`promoter`, `core_promoter`, `splice`)
5. **Yeni dataset olarak** `human_nontata_promoters` indir/CSV'ye çevir
6. DNABERT-2 eğitimlerini çalıştır
7. Sonuçları tek CSV ve pivot dosyalarına topla

Not: `human_nontata_promoters` promoter task’i ile uyumludur (yeni task tipi değildir).

In [ ]:
# 0) Upload project.zip
from google.colab import files
files.upload()

In [ ]:
# 1) Unzip and enter project
import os, zipfile, shutil

ZIP = '/content/project.zip'
PROJECT_DIR = '/content/project'

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

with zipfile.ZipFile(ZIP, 'r') as zf:
    zf.extractall('/content')

print('Exists:', os.path.exists(PROJECT_DIR))
print('Project files:', sorted(os.listdir(PROJECT_DIR))[:20])

In [ ]:
# 2) Install dependencies (stable set)
%cd /content/project
!pip -q install -r requirements.txt
!pip -q uninstall -y peft || true
!pip -q install genomic-benchmarks==0.0.9

In [ ]:
# 3) Environment check
%cd /content/project
!python scripts/check_environment.py

In [ ]:
# 4) Download GUE
%cd /content/project
!python scripts/download_gue.py --out-dir data/raw

In [ ]:
# 5) Build low-data splits for GUE tasks
%cd /content/project

RATIOS = '0.01,0.10,1.0'
SEEDS = '13'

# Promoter Detection
!python scripts/build_low_data_splits.py \
  --source-dir data/raw/GUE/prom/prom_300_all \
  --output-root data/low_data/prog2/prom_300_all \
  --ratios $RATIOS \
  --seeds $SEEDS

# Core Promoter Detection
!python scripts/build_low_data_splits.py \
  --source-dir data/raw/GUE/prom/prom_core_all \
  --output-root data/low_data/prog2/prom_core_all \
  --ratios $RATIOS \
  --seeds $SEEDS

# Splice Site Prediction
!python scripts/build_low_data_splits.py \
  --source-dir data/raw/GUE/splice/reconstructed \
  --output-root data/low_data/prog2/splice_reconstructed \
  --ratios $RATIOS \
  --seeds $SEEDS

## External dataset (same task family): human_nontata_promoters

Bu veri, promoter / non-promoter ikili sınıflandırmasıdır.
Burada **OCR task'i yok**.

In [ ]:
# 6) Download + convert human_nontata_promoters -> train/dev/test CSV
%cd /content/project

from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from genomic_benchmarks.loc2seq import download_dataset

dest = Path('/content/gb_data')
dest.mkdir(parents=True, exist_ok=True)

# downloads folder structure like: /content/gb_data/human_nontata_promoters/train/<class>/*.txt
root = download_dataset('human_nontata_promoters', version=0, dest_path=dest, force_download=False)
root = Path(root)
print('Downloaded root:', root)

def load_split(split_name):
    split_dir = root / split_name
    class_dirs = sorted([d for d in split_dir.iterdir() if d.is_dir()])
    label_map = {d.name: i for i, d in enumerate(class_dirs)}

    rows = []
    for cls_dir in class_dirs:
        label_id = label_map[cls_dir.name]
        for fp in cls_dir.glob('*.txt'):
            seq = fp.read_text().strip().upper()
            rows.append({'sequence': seq, 'label': label_id})

    df = pd.DataFrame(rows)
    return df, label_map

train_df, label_map = load_split('train')
test_df, _ = load_split('test')

train_df, dev_df = train_test_split(
    train_df,
    test_size=0.10,
    random_state=13,
    stratify=train_df['label']
)

out = Path('/content/project/data/raw/extra/human_nontata_promoters')
out.mkdir(parents=True, exist_ok=True)
train_df.to_csv(out / 'train.csv', index=False)
dev_df.to_csv(out / 'dev.csv', index=False)
test_df.to_csv(out / 'test.csv', index=False)

print('Saved:', out)
print('Label map:', label_map)
print('train/dev/test:', len(train_df), len(dev_df), len(test_df))
print(train_df.head())

In [ ]:
# 7) Build low-data splits for external promoter dataset
%cd /content/project
RATIOS = '0.01,0.10,1.0'
SEEDS = '13'

!python scripts/build_low_data_splits.py \
  --source-dir data/raw/extra/human_nontata_promoters \
  --output-root data/low_data/prog2/human_nontata_promoters \
  --ratios $RATIOS \
  --seeds $SEEDS

In [ ]:
# 8) Smoke test (short run) on external promoter dataset
%cd /content/project
!python scripts/train_dnabert2.py \
  --model_name_or_path zhihan1996/DNABERT-2-117M \
  --data_path data/low_data/prog2/human_nontata_promoters/r1_seed13 \
  --run_name smoke_nontata_r1_seed13 \
  --output_dir outputs/smoke_test_nontata \
  --seed 13 \
  --model_max_length 251 \
  --max_steps 60 \
  --per_device_train_batch_size 8 \
  --per_device_eval_batch_size 16 \
  --gradient_accumulation_steps 1 \
  --learning_rate 3e-5 \
  --warmup_steps 5 \
  --weight_decay 0.01 \
  --evaluation_strategy steps \
  --save_strategy steps \
  --eval_steps 60 \
  --save_steps 60 \
  --logging_steps 20 \
  --load_best_model_at_end True \
  --metric_for_best_model eval_f1_macro \
  --greater_is_better True \
  --save_total_limit 1 \
  --report_to none \
  --fp16 True \
  --use_lora False

## Main runs
Varsayılan grid:
- tasks: promoter, core_promoter, splice, promoter_external_nontata
- ratios: r1, r10, r100
- seed: 13

İstersen sadece promoter odaklı koşu için aşağıdaki `TASKS_TO_RUN` listesini daraltabilirsin.

In [ ]:
# 9) Main runs
%cd /content/project
python - << 'PY'
import subprocess, os

MODEL = 'zhihan1996/DNABERT-2-117M'
BASE_OUT = 'outputs/prog2/main_runs'

TASKS = [
    # task_name, low_data_root, model_max_length, max_steps
    ('promoter',                 'data/low_data/prog2/prom_300_all',             300, 600),
    ('core_promoter',            'data/low_data/prog2/prom_core_all',              70, 600),
    ('splice',                   'data/low_data/prog2/splice_reconstructed',      400, 600),
    ('promoter_external_nontata','data/low_data/prog2/human_nontata_promoters',   251, 600),
]

# If you want only promoter datasets, uncomment this:
# TASKS = [t for t in TASKS if t[0] in ['promoter', 'promoter_external_nontata']]

ratios = ['r1_seed13', 'r10_seed13', 'r100_seed13']

for task_name, root, max_len, max_steps in TASKS:
    for r in ratios:
        data_path = f'{root}/{r}'
        run_name = f'{task_name}_{r}'
        out_dir  = f'{BASE_OUT}/{task_name}/{r}'
        os.makedirs(out_dir, exist_ok=True)

        cmd = [
            'python', 'scripts/train_dnabert2.py',
            '--model_name_or_path', MODEL,
            '--data_path', data_path,
            '--run_name', run_name,
            '--output_dir', out_dir,
            '--seed', '13',
            '--model_max_length', str(max_len),
            '--max_steps', str(max_steps),
            '--per_device_train_batch_size', '8',
            '--per_device_eval_batch_size', '16',
            '--gradient_accumulation_steps', '1',
            '--learning_rate', '3e-5',
            '--warmup_steps', '50',
            '--weight_decay', '0.01',
            '--evaluation_strategy', 'steps',
            '--save_strategy', 'steps',
            '--eval_steps', '200',
            '--save_steps', '200',
            '--logging_steps', '50',
            '--load_best_model_at_end', 'True',
            '--metric_for_best_model', 'eval_f1_macro',
            '--greater_is_better', 'True',
            '--save_total_limit', '1',
            '--report_to', 'none',
            '--fp16', 'True',
            '--use_lora', 'False'
        ]

        print('\nRUN:', ' '.join(cmd))
        subprocess.run(cmd, check=True)

print('\nAll main runs finished.')
PY

In [ ]:
# 10) Aggregate all eval_results.json files
%cd /content/project
!python scripts/aggregate_results.py \
  --results-root outputs/prog2/main_runs \
  --out-csv outputs/prog2_summary_all_runs.csv \
  --out-mean-csv outputs/prog2_summary_mean_std.csv

!python - << 'PY'
import pandas as pd
from pathlib import Path
p = Path('outputs/prog2_summary_all_runs.csv')
print('exists:', p.exists())
if p.exists():
    df = pd.read_csv(p)
    print(df.head(10))
    print('rows:', len(df))
PY

In [ ]:
# 11) Make quick pivot tables for slides
%cd /content/project
python - << 'PY'
import pandas as pd

df = pd.read_csv('outputs/prog2_summary_all_runs.csv')

def parse_task_ratio(name):
    if name.startswith('core_promoter'):
        task = 'core_promoter'
    elif name.startswith('promoter_external_nontata'):
        task = 'promoter_external_nontata'
    elif name.startswith('promoter'):
        task = 'promoter'
    elif name.startswith('splice'):
        task = 'splice'
    else:
        task = 'other'

    ratio = 'na'
    for p in name.split('_'):
        if p.startswith('r') and p[1:].isdigit():
            ratio = p
            break
    return task, ratio

df[['task','ratio']] = df['run_name'].apply(lambda x: pd.Series(parse_task_ratio(x)))

pivot_f1 = df.pivot_table(index='task', columns='ratio', values='eval_f1_macro', aggfunc='mean')
pivot_auc = df.pivot_table(index='task', columns='ratio', values='eval_auroc', aggfunc='mean')

print('F1-macro pivot:')
print(pivot_f1)
print('\nAUROC pivot:')
print(pivot_auc)

pivot_f1.to_csv('outputs/prog2_f1_pivot.csv')
pivot_auc.to_csv('outputs/prog2_auroc_pivot.csv')
print('\nSaved: outputs/prog2_f1_pivot.csv, outputs/prog2_auroc_pivot.csv')
PY

In [ ]:
# 12) Export outputs zip
%cd /content/project
!zip -r /content/prog2_outputs_nontata.zip outputs/prog2/main_runs outputs/prog2_summary_all_runs.csv outputs/prog2_summary_mean_std.csv outputs/prog2_f1_pivot.csv outputs/prog2_auroc_pivot.csv